
# 🎧 EDA Visual con Seaborn — *Spotify Tracks 2023* (Plantilla semi-guiada)

**Objetivo:** practicar EDA visual con **Seaborn** sobre un dataset moderno de Spotify (2023).  
Trabajaremos distribuciones, relaciones, comparaciones por género y correlaciones, concluyendo con una **mini‑historia visual**.

**Librerías:** `pandas`, `seaborn`, `matplotlib`  
**Estilo base:** `sns.set_theme(style="whitegrid", palette="muted")`

> ℹ️ Esta plantilla es *semi‑guiada*: hay celdas con **“# TODO”** y **pistas 💡**. Ejecuta paso a paso y completa lo que falte.



## 1) Preparación de entorno y descarga automática del dataset

Ejecuta esta celda. Si una URL falla, el script probará otra. Como **último recurso**, podrás subir el CSV manualmente.


In [1]:

# %pip -q install pandas seaborn matplotlib

import pandas as pd, seaborn as sns, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

URLS = [
    "https://raw.githubusercontent.com/yogesh-virat/Spotify-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/abdullahfawzy/Spotify-Dataset-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/raulgomez99/spotify-eda-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/LuAnhThe/Spotify-2023/refs/heads/main/spotify-2023.csv"
]

df, last_err = None, None
for u in URLS:
    try:
        print(f"Intentando descargar: {u}")
        df = pd.read_csv(u)
        print("✅ Descargado con éxito.")
        break
    except Exception as e:
        last_err = e
        print(f"⚠️ Falló: {e}")

if df is None:
    print("\n❌ No se pudo descargar automáticamente. Sube el CSV manualmente en la celda siguiente.")
else:
    display(df.head(3))
    print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")


Intentando descargar: https://raw.githubusercontent.com/yogesh-virat/Spotify-2023/main/spotify-2023.csv
⚠️ Falló: HTTP Error 404: Not Found
Intentando descargar: https://raw.githubusercontent.com/abdullahfawzy/Spotify-Dataset-2023/main/spotify-2023.csv
⚠️ Falló: HTTP Error 404: Not Found
Intentando descargar: https://raw.githubusercontent.com/raulgomez99/spotify-eda-2023/main/spotify-2023.csv
⚠️ Falló: HTTP Error 404: Not Found
Intentando descargar: https://raw.githubusercontent.com/LuAnhThe/Spotify-2023/refs/heads/main/spotify-2023.csv
✅ Descargado con éxito.


,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6


Filas: 952 | Columnas: 24


In [2]:
df = pd.read_csv("./Seaborn/spotify-2023_2.csv")  # Subir manualmente si la descarga automática falló
# Cuidado con el path en vuestro ordenador o en Coogle Colab
# convertir a parquet
df.to_parquet("./Seaborn/spotify-2023.parquet")


### (Opcional) Si la descarga falló, sube el CSV manualmente


In [ ]:

# from google.colab import files
# uploaded = files.upload()
# import io
# df = pd.read_csv(io.BytesIO(uploaded[next(iter(uploaded))]))
# display(df.head()); print(df.shape)



## 2) Exploración inicial y preparación mínima
1. Vista rápida de datos, tipos y valores nulos.  
2. Selección de **columnas relevantes** para el análisis visual.


In [4]:

assert 'df' in globals() and df is not None, "Primero descarga o carga el dataset en la celda anterior."

_ = df.info()
display(df.describe(include='all').transpose().head(12))

def normalize(col):
    return col.strip().lower().replace(' ', '_').replace('%','perc').replace('(','').replace(')','').replace('-','_')
df.columns = [normalize(c) for c in df.columns]

def first_existing(names):
    for n in names:
        if n in df.columns:
            return n
    return None

col_pop   = first_existing(['popularity','popularity_score'])
col_dance = first_existing(['danceability_perc','danceability'])
col_energy= first_existing(['energy_perc','energy'])
col_val   = first_existing(['valence_perc','valence'])
col_tempo = first_existing(['tempo'])
col_loud  = first_existing(['loudness_db','loudness'])
col_genre = first_existing(['genre','track_genre'])
col_artist= first_existing(['artist_name','artist','artists'])
track_name= first_existing(['track_name','track','song_name','song'])

selected_cols = [c for c in [col_pop,col_dance,col_energy,col_val,col_tempo,col_loud,col_genre,col_artist,track_name] if c]
print("Columnas detectadas:", selected_cols)

df_sel = df[selected_cols].copy()
display(df_sel.head(5))
display(df_sel.isna().sum())
display(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   unnamed:_0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
unnamed:_0,114000.0,NaN,NaN,NaN,56999.5,32909.109681,0.0,28499.75,56999.5,85499.25,113999.0
track_id,114000,89741,6S3JlDAGk3uu3NtZbPnuhS,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
artists,113999,31437,The Beatles,279,NaN,NaN,NaN,NaN,NaN,NaN,NaN
album_name,113999,46589,Alternative Christmas 2022,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_name,113999,73608,Run Rudolph Run,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popularity,114000.0,NaN,NaN,NaN,33.238535,22.305078,0.0,17.0,35.0,50.0,100.0
duration_ms,114000.0,NaN,NaN,NaN,228029.153114,107297.712645,0.0,174066.0,212906.0,261506.0,5237295.0
explicit,114000,2,False,104253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
danceability,114000.0,NaN,NaN,NaN,0.5668,0.173542,0.0,0.456,0.58,0.695,0.985
energy,114000.0,NaN,NaN,NaN,0.641383,0.251529,0.0,0.472,0.685,0.854,1.0


Columnas detectadas: ['popularity', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'track_genre', 'artists', 'track_name']


,popularity,danceability,energy,valence,tempo,loudness,track_genre,artists,track_name
0,73,0.676,0.4610,0.715,87.917,-6.746,acoustic,Gen Hoshino,Comedy
1,55,0.420,0.1660,0.267,77.489,-17.235,acoustic,Ben Woodward,Ghost - Acoustic
2,57,0.438,0.3590,0.120,76.332,-9.734,acoustic,Ingrid Michaelson;ZAYN,To Begin Again
3,71,0.266,0.0596,0.143,181.740,-18.515,acoustic,Kina Grannis,Can't Help Falling In Love
4,82,0.618,0.4430,0.167,119.949,-9.681,acoustic,Chord Overstreet,Hold On


popularity      0
danceability    0
energy          0
valence         0
tempo           0
loudness        0
track_genre     0
artists         1
track_name      1
dtype: int64

['unnamed:_0',
 'track_id',
 'artists',
 'album_name',
 'track_name',
 'popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

In [ ]:
#localizar la fila con nombre de artista nulo, si existe, poner el número de fila y los datos de esa fila
null_artist_rows = df_sel[df_sel[col_artist].isna()]
print("Filas con artista nulo:")
for index, row in null_artist_rows.iterrows():
    print(f"Fila {index}: {row.to_dict()}")
print(f"Fila con artista nulo: {df.loc[null_artist_rows.index]}")


## 3) Gráficos univariantes (distribuciones)
> 💡 **Objetivo:** conocer la forma de la distribución de métricas clave (popularidad, danceability, energy, tempo).


In [ ]:

import numpy as np
targets = [c for c in [col_pop, col_dance, col_energy, col_tempo] if c]

fig, axes = plt.subplots(1, len(targets), figsize=(4*len(targets), 3))
if len(targets) == 1:
    axes = [axes]

for ax, col in zip(axes, targets):
    sns.histplot(data=df_sel, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribución de {col}")
plt.tight_layout(); plt.show()

# 🧩 TODO: crea un boxplot de 'popularity' por 'genre' para los 8 géneros con más canciones.
# Pista 💡: agrupa por género, ordena por tamaño y filtra el top 8 antes de graficar.



## 4) Relaciones bivariantes (¿qué explica la popularidad?)
> 💡 Explora relaciones entre **popularidad** y otras variables cuantitativas.


In [ ]:

pairs = [c for c in [col_dance, col_energy, col_tempo, col_loud, col_val] if c]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4))
if len(pairs) == 1: axes = [axes]

for ax, col in zip(axes, pairs):
    sns.scatterplot(data=df_sel.sample(min(5000, len(df_sel))), x=col, y=col_pop, alpha=0.3, ax=ax)
    ax.set_title(f"{col_pop} vs {col}")
plt.tight_layout(); plt.show()

# 🧩 TODO: usa sns.lmplot para ver tendencia lineal (hue=col_genre si es manejable).
# Pista 💡: samplea 10-15 géneros más frecuentes.



## 5) Comparaciones por género (categóricas)
> 💡 Compara popularidad y rasgos (danceability/energy) **entre géneros**.


In [ ]:

import numpy as np

if col_genre is not None:
    top_genres = df_sel[col_genre].value_counts().head(8).index
    dfg = df_sel[df_sel[col_genre].isin(top_genres)].copy()

    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    sns.barplot(data=dfg, x=col_genre, y=col_pop, estimator=np.mean, errorbar='sd', ax=ax[0])
    ax[0].set_title("Popularidad media por género (top 8)")
    ax[0].tick_params(axis='x', rotation=30)

    metric = col_dance if col_dance else col_energy
    sns.violinplot(data=dfg, x=col_genre, y=metric, inner='quartile', ax=ax[1])
    ax[1].set_title(f"Distribución de {metric} por género (top 8)")
    ax[1].tick_params(axis='x', rotation=30)

    plt.tight_layout(); plt.show()

# 🧩 TODO: superpone stripplot sobre el violinplot (color='k', size=2, alpha=0.3).



## 6) Correlaciones y multivariante
> 💡 Identifica pares de variables con fuerte relación (positiva o negativa).


In [ ]:

num_cols = [c for c in [col_pop,col_dance,col_energy,col_val,col_tempo,col_loud] if c]
corr = df_sel[num_cols].corr(numeric_only=True)
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap="YlGnBu", vmin=-1, vmax=1, center=0)
plt.title("Matriz de correlaciones (métricas principales)")
plt.show()

# 🧩 TODO: crea un pairplot con 4-5 variables clave (usa hue=col_genre si el nº de géneros es pequeño).



## 7) Storytelling visual (mini‑historia)
Redacta **2-3 conclusiones** basadas en tus gráficos anteriores.



✍️ **Tu texto aquí:**

- …  
- …  
- …  



## 8) Evaluación práctica (entrega)
> **Tarea:** Crea un gráfico propio que explore la relación entre **popularidad** y el rasgo que elijas (*danceability, energy, valence, tempo, loudness* …).  
> Acompáñalo con una **mini‑conclusión (3‑5 líneas)**.


In [ ]:

# TODO: tu gráfico final aquí
# Ejemplo guía (comentado):
# sns.set_palette("pastel")
# dfg = df_sel[df_sel[col_genre].isin(df_sel[col_genre].value_counts().head(8).index)]
# sns.lmplot(data=dfg.sample(8000, random_state=42), x=col_dance, y=col_pop, hue=col_genre, height=4, aspect=1.4, scatter_kws={'alpha':0.2})
# plt.title("Popularidad vs Bailabilidad por género (muestra)")
# plt.show()



### Conclusión (3–5 líneas)
Escribe aquí tus observaciones sobre el gráfico anterior.
